# Flujo completo del trabajo — Predicción de demanda de transporte público en Madrid

**TFM · Cuaderno principal**

Este cuaderno recorre **el flujo completo** del trabajo, de los ficheros crudos a los gráficos
finales, en las seis etapas del guion:

1. Carga y comprobación de datos
2. Preparación de variables
3. Partición temporal
4. Ejecución de los modelos principales
5. Comparación de resultados
6. Gráficos finales

Toda la lógica vive en `src/`. Aquí **no se define ninguna función, ninguna figura y ninguna
métrica**: se importa, se ejecuta y se muestra. Es el mismo criterio que sigue
`06_dashboard.ipynb`, que es el **panel de resultados** (visualización interactiva sobre
artefactos ya persistidos); este cuaderno es el **flujo end-to-end** y se ejecuta sin widgets,
de principio a fin.

## Requisito previo

El paquete debe estar instalado en modo editable para que `from src...` resuelva desde
`notebooks/`:

```bash
pip install -e .
```

## Qué se recomputa y qué se carga

| Etapa | Política | Motivo |
|---|---|---|
| Ingestión, integridad, variables, partición | **recomputado** | determinista y en menos de un segundo |
| Baselines (persistencia, estacional, medias móviles) | **recomputado** | lecturas puras de columnas |
| SARIMAX | **recomputado** | se carga solo el *orden* `(2,1,2)x(0,1,2,7)` ya seleccionado por AIC; reajustarlo cuesta < 1 s |
| XGBoost residual (etapa 2) y XGBoost-alone | **recomputado** | rejilla de 54 candidatos, ~15 s cada uno, bit-exacto bajo `GLOBAL_SEED` |
| **LSTM (etapa 1)** | **cargada** | ver la justificación de la etapa 4 |

## Dos advertencias antes de ejecutar

**No escribe nada.** Todas las llamadas usan `save=False` o devuelven resultados en memoria.
Ejecutar este cuaderno no puede modificar ni corromper los artefactos de `data/` ni de
`models/`.

**Se verifica a sí mismo.** En la etapa 5, cada métrica recomputada se contrasta contra el
valor publicado en `data/processed/full_comparison.parquet` y el cuaderno **levanta una
excepción** si alguna se desvía más de la tolerancia. No afirma ser reproducible: lo demuestra
o falla.

## Sobre estas salidas

Las salidas guardadas en este fichero corresponden a una ejecución real completa
(`jupyter nbconvert --to notebook --execute --inplace`).
Los datos de esa ejecución se imprimen en la celda siguiente.

In [1]:
# El renderer de Plotly se deja en su valor por defecto a proposito: fijar
# pio.renderers.default = "notebook_connected" cuelga la ejecucion bajo nbconvert, que es
# como se generan las salidas guardadas de este cuaderno.
import json
import platform
import sys
import tempfile
import time
from datetime import datetime
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from IPython.display import display

from src.utils.seed import GLOBAL_SEED, set_global_seed

# deterministic_ops=False: no se entrena ninguna red neuronal en este cuaderno (la etapa 1 se
# carga), asi que activar el determinismo de operaciones de TensorFlow solo anadiria coste.
set_global_seed(GLOBAL_SEED, deterministic_ops=False)

T0 = time.perf_counter()

import statsmodels
import plotly
import xgboost

print(f"ejecutado          : {datetime.now():%Y-%m-%d %H:%M}")
print(f"python             : {sys.version.split()[0]}  ({platform.system()} {platform.machine()})")
print(f"numpy              : {np.__version__}")
print(f"pandas             : {pd.__version__}")
print(f"xgboost            : {xgboost.__version__}")
print(f"statsmodels        : {statsmodels.__version__}")
print(f"plotly             : {plotly.__version__}")
print(f"GLOBAL_SEED        : {GLOBAL_SEED}")

ejecutado          : 2026-09-09 12:04
python             : 3.13.5  (Windows AMD64)
numpy              : 2.2.6
pandas             : 2.3.3
xgboost            : 3.0.5
statsmodels        : 0.14.6
plotly             : 6.3.0
GLOBAL_SEED        : 42


---

# 1. Carga y comprobación de datos

Tres fuentes crudas, cada una con su trampa de formato documentada en `src/ingestion/`:

| Fuente | Fichero | Trampa |
|---|---|---|
| Demanda CRTM | `CRTM_Evolucion_demanda_diaria.xlsx` | hoja `diaria`, **`header=1`**; con `header=2` se pierde 2023-01-01 |
| Meteorología | `open-meteo-40.39N3.68W666m.csv` | **`skiprows=3`**; los metadatos declaran zona `Europe/Berlin` |
| Calendario laboral | `300082-1-calendario_laboral-csv.csv` | `sep=";"`, `encoding="utf-8-sig"`, fechas `%d/%m/%Y`, `Dia_semana` sin acentos |

La unificación (`unify_frames`) es deliberadamente paranoica. Un `inner join` sobre `date` es
la única unión segura —un `left join` fabricaría filas exógenas todo-NaN—, pero un `inner
join` **también oculta los huecos de cobertura por construcción**. Por eso cada invariante que
la unión habría tragado en silencio está escrita como un `raise` que nombra las fechas
culpables.

In [2]:
from src.ingestion.load_calendar import load_calendar
from src.ingestion.load_crtm import load_crtm
from src.ingestion.load_weather import load_weather
from src.ingestion.unify import integrity_report, print_schema_summary, unify

crtm, weather, calendar = load_crtm(), load_weather(), load_calendar()
for nombre, frame in (("CRTM", crtm), ("meteo", weather), ("calendario", calendar)):
    print(f"{nombre:<11}: {frame.shape[0]:>5} filas x {frame.shape[1]:>2} columnas   "
          f"{frame['date'].min():%Y-%m-%d} -> {frame['date'].max():%Y-%m-%d}")

# save=False: el cuaderno no reescribe data/interim/unified_daily.parquet.
unified = unify(save=False)
print()
print_schema_summary(unified)

CRTM       :  1310 filas x  6 columnas   2023-01-01 -> 2026-08-02
meteo      :  1310 filas x 22 columnas   2023-01-01 -> 2026-08-02
calendario :  1310 filas x  5 columnas   2023-01-01 -> 2026-08-02

UNIFIED DAILY DATASET — schema summary
rows        : 1310
columns     : 31
date range  : 2023-01-01 -> 2026-08-02 (daily, 1310 observations)
missing days: 0
duplicates  : 0
total nulls : 0
------------------------------------------------------------------------
  #  column                             dtype                nulls
------------------------------------------------------------------------
  0  date                               datetime64[ns]       0
  1  metro                              int64                0
  2  emt                                int64                0
  3  carretera                          int64                0
  4  cercanias                          int64                0
  5  total                              int64                0
  6  temperature_2m_m

In [3]:
display(crtm.head(3))

# Las tres primeras comprobaciones ya son `raise` dentro de unify_frames: si este frame existe,
# es que pasaron. Se muestran con su valor observado porque una guardia que no salta es
# invisible, y el objetivo aqui es *ensenar* que los invariantes se cumplen, no afirmarlo.
# La cuarta -- metro+emt+carretera+cercanias == total -- es la razon por la que las columnas de
# operador del mismo dia son fuga perfecta del objetivo.
print("INFORME DE INTEGRIDAD")
display(integrity_report(unified))

,date,metro,emt,carretera,cercanias,total
0,2023-01-01,685684,319488,155714,174991,1335877
1,2023-01-02,1581661,1024836,588003,446467,3640967
2,2023-01-03,1781186,1151845,662751,510268,4106050


INFORME DE INTEGRIDAD


,comprobacion,esperado,observado,veredicto
0,filas,1310,1310,OK
1,huecos de calendario,0,0,OK
2,fechas duplicadas,0,0,OK
3,valores nulos,0,0,OK
4,suma de operadores == total,desviacion maxima 0,desviacion maxima 0,OK


In [4]:
# La guardia de `day_type` no acepta valores desconocidos en silencio: un tipo de dia no
# previsto seria una fila one-hot todo-ceros, que es un fallo mudo. Se comprueba disparandola.
from src.features.calendar_features import add_calendar_features

corrupto = unified.copy()
corrupto["day_type"] = corrupto["day_type"].astype(str)
corrupto.loc[0, "day_type"] = "fiesta_inventada"

try:
    add_calendar_features(corrupto)
except ValueError as exc:
    print("GUARDIA DISPARADA (esperado):")
    print(f"  {exc}")
else:
    raise AssertionError("la guardia de day_type no salto")

GUARDIA DISPARADA (esperado):
  calendar_features: column 'day_type' contains unmapped value(s) ['fiesta_inventada']. Expected one of ['laborable', 'sabado', 'domingo', 'festivo', 'domingo festivo']. Extend the category list deliberately rather than emitting an all-zero row.


---

# 2. Preparación de variables

Cinco bloques, ensamblados por `src/features/build_features.py`:

| Bloque | Módulo | Contenido |
|---|---|---|
| Calendario | `calendar_features.py` | one-hot de `day_type` y `holiday_type`, fin de semana, día puente |
| Fourier | `fourier_features.py` | armónicos semanal (k=1) y anual (k=1,2), función determinista de la fecha |
| Retardos | `lag_features.py` | `total` y los cuatro operadores en los retardos 1, 2, 3, 7, 14, 21, 28 |
| Móviles | `rolling_features.py` | media y desviación de `total` en ventanas de 7 y 28 días |
| Meteorología | `weather_features.py` | 21 variables en retardos 1, 2, 3, 7 más media móvil de 7 |

Dos reglas gobiernan el bloque y ninguna es negociable:

**`shift(1)` antes de `rolling`, nunca al revés.** Una media móvil sin desplazar incluye el
día que se quiere predecir. `rolling_features.py` desplaza primero y fija
`min_periods == window`, de modo que jamás emite una ventana parcial.

**La meteorología del mismo día no entra en la matriz.** Es dato observado, no previsión;
usarla sin retardar supone conocimiento perfecto del futuro e infla el rendimiento aparente.
Las 21 columnas crudas se conservan como referencia para EDA, fuera de `feature_columns()`.

Las 28 primeras filas arrastran NaN de las ventanas de 28 días. **Se dejan en su sitio**:
descartarlas o imputarlas es una decisión que depende del consumidor (la LSTM necesita una
ventana contigua sin NaN; XGBoost los maneja de forma nativa), así que pertenece a la etapa de
modelado, no a esta.

In [5]:
from src.evaluation.dashboard_data import classify_feature
from src.features.build_features import build_features, feature_columns, print_feature_report

features = build_features(unified)
feat_cols = feature_columns(features)
print(f"filas             : {len(features)}   (sin cambio respecto a la unificacion)")
print(f"variables feat_   : {len(feat_cols)}")
print()

grupos = pd.Series([classify_feature(c) for c in feat_cols]).value_counts()
display(grupos.rename_axis("bloque").to_frame("n variables"))

filas             : 1310   (sin cambio respecto a la unificacion)
variables feat_   : 161



,n variables
bloque,
weather,105
lag_operator,28
calendar,11
lag_total,7
fourier_annual,4
rolling,4
fourier_weekly,2


In [6]:
print_feature_report(features)

FEATURE TABLE — Phase 2 report
rows              : 1310  (unchanged from data/interim)
columns total     : 192
engineered feats  : 161   (prefix 'feat_')
raw reference     : 8
raw weather (ref) : 21  (same-day, NOT in the model matrix)
date range        : 2023-01-01 -> 2026-08-02
------------------------------------------------------------------------------
feature                                      dtype      NaNs  first valid
------------------------------------------------------------------------------
feat_day_type_laborable                      int8          0  2023-01-01
feat_day_type_sabado                         int8          0  2023-01-01
feat_day_type_domingo                        int8          0  2023-01-01
feat_day_type_festivo                        int8          0  2023-01-01
feat_day_type_domingo_festivo                int8          0  2023-01-01
feat_holiday_type_none                       int8          0  2023-01-01
feat_holiday_type_festivo_nacional           int8

## Las guardias anti-fuga no son comentarios: son `raise`

CLAUDE.md fija cinco reglas anti-fuga y el proyecto las implementa como excepciones, no como
avisos ni como notas en un docstring. La diferencia importa: un aviso se ignora en una salida
larga; una excepción detiene el pipeline.

Aquí se **disparan de verdad**, inyectando en cada caso la violación que deben detectar.

Conviene ser preciso sobre qué demuestra cada una, porque hay dos niveles de protección y el
segundo es más fuerte que el primero:

- Las guardias de **operador del mismo día** y de **meteorología del mismo día** inspeccionan
  el bloque de variables ya construido. Para dispararlas hay que contaminar ese bloque
  directamente, porque **los constructores públicos no permiten llegar hasta ellas**:
  `add_lag_features` sólo emite retardos de 1 día en adelante y `add_weather_features` sólo
  emite formas retardadas o móviles, de modo que una columna del mismo día no tiene por dónde
  entrar. La guardia es la red de seguridad; la imposibilidad estructural es la garantía real.
- La guardia de **orden temporal** sí se dispara desde la API pública, sin más que pasar un
  frame desordenado.

In [7]:
from src.features.lag_features import add_lag_features
from src.features.weather_features import add_weather_features

# Se importan las guardias directamente: son la implementacion real de la proteccion, y la
# unica forma de inyectarles una violacion que los constructores publicos hacen inalcanzable.
from src.features.lag_features import _assert_no_same_day_leakage, add_lag_features
from src.features.weather_features import _assert_no_same_day_weather, add_weather_features

bloque_lags = add_lag_features(unified)
bloque_meteo = add_weather_features(unified)

# (a) Columna de operador del mismo dia colada en el bloque de retardos.
#     metro+emt+carretera+cercanias == total exactamente, luego es fuga perfecta del objetivo.
try:
    _assert_no_same_day_leakage(bloque_lags.assign(metro=unified["metro"].to_numpy()))
except ValueError as exc:
    print("(a) GUARDIA DE FUGA — operador del mismo dia:")
    print(f"    {exc}")
    print()
else:
    raise AssertionError("la guardia de operador del mismo dia no salto")

# (b) Meteorologia del mismo dia, sin retardar, colada en el bloque meteorologico.
try:
    _assert_no_same_day_weather(
        bloque_meteo.assign(
            temperature_2m_mean=unified["temperature_2m_mean"].to_numpy()
        )
    )
except ValueError as exc:
    print("(b) GUARDIA DE METEOROLOGIA DEL MISMO DIA:")
    print(f"    {exc}")
    print()
else:
    raise AssertionError("la guardia de meteorologia del mismo dia no salto")

# (c) Filas desordenadas: .shift() dejaria de ser un retardo de calendario. Esta si se
#     dispara desde la API publica.
try:
    add_lag_features(unified.sample(frac=1.0, random_state=GLOBAL_SEED))
except ValueError as exc:
    print("(c) GUARDIA DE ORDEN TEMPORAL:")
    print(f"    {exc}")
else:
    raise AssertionError("la guardia de orden temporal no salto")

(a) GUARDIA DE FUGA — operador del mismo dia:
    LEAKAGE GUARD: raw same-day column(s) ['metro'] present in the lag feature output. metro+emt+carretera+cercanias == total exactly, so a same-day operator column is perfect leakage of the target.

(b) GUARDIA DE METEOROLOGIA DEL MISMO DIA:
    SAME-DAY WEATHER GUARD: unlagged weather column(s) ['temperature_2m_mean'] present in the feature output. Same-day weather is observed data, not a forecast; using it assumes perfect foreknowledge and inflates apparent performance.

(c) GUARDIA DE ORDEN TEMPORAL:
    lag_features: frame is not sorted ascending by date; .shift() would draw values from the wrong rows.


---

# 3. Partición temporal

`src/utils/splits.py` es la **única fuente de verdad**. Si la LSTM y los baselines
discreparan un solo día sobre dónde empieza el test, sus métricas no serían comparables y la
tabla maestra entera perdería sentido.

**70/15/15 estrictamente cronológico**, sin barajar. El redondeo se resuelve sobre fracciones
**acumuladas** (`floor(0,70n)`, `floor(0,85n)`) y no por partición: con n = 1310,
0,15·n = 196,5, y redondear cada bloque por separado perdería o duplicaría un día en la
frontera. El día sobrante va al test.

**La política de warm-up descarta 28 filas, y sólo del bloque de entrenamiento.** Si la
ventana de warm-up alcanzase alguna vez a validación o test, el resultado silencioso sería un
periodo de evaluación acortado y unas métricas no comparables entre modelos ni entre
ejecuciones. Por eso `apply_warmup_policy` lo comprueba y **levanta** en lugar de encogerse de
hombros.

In [8]:
from src.utils.splits import apply_warmup_policy, chronological_split, split_masks

bounds = chronological_split(features["date"])
print(bounds.describe())

trimmed, dropped = apply_warmup_policy(features, bounds)
trimmed = trimmed.reset_index(drop=True)
masks = split_masks(trimmed, bounds)

print()
print(f"warm-up descartado : {dropped} filas "
      f"({features['date'].iloc[0]:%Y-%m-%d} -> {features['date'].iloc[dropped - 1]:%Y-%m-%d}), "
      "todas dentro de entrenamiento")
print(f"filas tras el trim : {len(trimmed)} (eran {len(features)})")
for nombre, mask in masks.items():
    print(f"  {nombre:<6}: {int(mask.sum()):>4} filas")

# Las fronteras estan fijadas en CLAUDE.md; si cambiaran, todo lo publicado dejaria de ser
# comparable, asi que se comprueban aqui en lugar de darse por supuestas.
assert bounds.as_dict() == {
    "train_start": "2023-01-01", "train_end": "2025-07-05",
    "val_start": "2025-07-06", "val_end": "2026-01-17",
    "test_start": "2026-01-18", "test_end": "2026-08-02",
}, "las fronteras de particion no coinciden con las documentadas en CLAUDE.md"
assert (bounds.n_train, bounds.n_val, bounds.n_test) == (917, 196, 197)
assert int(masks["train"].sum()) == 889
print("\nfronteras y recuentos conformes con CLAUDE.md")

train 2023-01-01 -> 2025-07-05  ( 917 d, 70.0%)
val   2025-07-06 -> 2026-01-17  ( 196 d, 15.0%)
test  2026-01-18 -> 2026-08-02  ( 197 d, 15.0%)

warm-up descartado : 28 filas (2023-01-01 -> 2023-01-28), todas dentro de entrenamiento
filas tras el trim : 1282 (eran 1310)
  train :  889 filas
  val   :  196 filas
  test  :  197 filas

fronteras y recuentos conformes con CLAUDE.md


In [9]:
from src.evaluation.memoria_figures import splits_figure

display(splits_figure(features))

---

# 4. Ejecución de los modelos principales

## Qué se recomputa aquí y qué se carga, y por qué

Se **recomputan en vivo**, bajo `GLOBAL_SEED = 42` y con las versiones fijadas en
`requirements.txt`: los cuatro baselines, SARIMAX, el XGBoost de la etapa 2 (residuos) y el
XGBoost-alone de control. Todos son deterministas y reproducen sus artefactos publicados.

De SARIMAX se carga únicamente el **orden** `(2,1,2)x(0,1,2,7)` que la búsqueda AIC
seleccionó, persistido en `sarimax_selected_order.json`. Esa búsqueda en rejilla es lo que
tarda unos 6 minutos; reajustar el modelo en el orden ya elegido cuesta menos de un segundo, y
reproduce `baseline_predictions.parquet` bit a bit.

## La etapa 1 (LSTM) se carga, y es la decisión correcta, no un atajo

`models/lstm_stage1_final.keras` y sus predicciones OOF se leen de disco. **No se reentrena la
red.** El motivo es metodológico antes que práctico:

> CLAUDE.md establece que la etapa 2 debe corregir los residuos del **mismo** modelo de etapa 1
> que produjo los resultados publicados. El docstring de
> `src/models/hybrid_residual/combine.py` lo dice sin rodeos: reentrenar la red
> *"introduciría un segundo modelo de etapa 1 cuyas salidas difieren de aquel con el que se
> emparejó el modelo residual"*.

Si este cuaderno reentrenase la LSTM, calcularía los residuos `e = y − ŷ_LSTM` **contra un
modelo distinto del que reporta la memoria**, y la comparación de la etapa 5 dejaría de medir
el pipeline publicado para medir la diferencia entre dos redes. A ello se suma que el
determinismo de las operaciones de red neuronal no está garantizado entre máquinas, de modo
que ni siquiera con la misma semilla se puede prometer la misma red en el ordenador de otra
persona.

El artefacto **es** el modelo de referencia. Cargarlo es lo que preserva la identidad de la
etapa 1; recomputarlo la rompería.

In [10]:
from src.models.baselines.run_baselines import SARIMAX_ORDER_FILE, run
from src.models.baselines.sarimax import SarimaxOrder

# Se carga el ORDEN seleccionado por AIC, no el modelo: la busqueda en rejilla (~6 min) ya se
# ejecuto y su resultado esta persistido. El ajuste en ese orden se rehace aqui en vivo.
seleccion = json.loads(Path(SARIMAX_ORDER_FILE).read_text(encoding="utf-8"))
orden = SarimaxOrder(
    tuple(seleccion["order"]), tuple(seleccion["seasonal_order"]), seleccion["aic"]
)
print(f"orden SARIMAX cargado de disco: {orden}\n")

t = time.perf_counter()
baseline_preds, baseline_tables, _ = run(cached_order=orden)
print(f"\n[{time.perf_counter() - t:.1f} s] baselines + SARIMAX recomputados")

orden SARIMAX cargado de disco: SARIMAX(2, 1, 2)x(0, 1, 2, 7) (AIC=24,616.89)

CHRONOLOGICAL SPLIT (70/15/15, single source of truth: src/utils/splits.py)
train 2023-01-01 -> 2025-07-05  ( 917 d, 70.0%)
val   2025-07-06 -> 2026-01-17  ( 196 d, 15.0%)
test  2026-01-18 -> 2026-08-02  ( 197 d, 15.0%)

warm-up policy    : dropped 28 row(s) (2023-01-01 -> 2023-01-28), all inside the training block
rows after trim   : 1282 (was 1310)
  train :  889 rows
  val   :  196 rows
  test  :  197 rows

SARIMAX — grid search over (p,d,q)(P,D,Q,7), AIC on TRAINING SET ONLY
using cached order: SARIMAX(2, 1, 2)x(0, 1, 2, 7) (AIC=24,616.89)

exog (5 calendar columns, no lag features):
  - feat_is_weekend
  - feat_is_bridge_day
  - feat_holiday_type_festivo_nacional
  - feat_holiday_type_festivo_de_la_comunidad_de_madrid
  - feat_holiday_type_festivo_local_de_la_ciudad_de_madrid



[1.0 s] baselines + SARIMAX recomputados


In [11]:
from src.evaluation.metrics import format_table

for split in ("train", "val", "test"):
    print("=" * 96)
    print(f"BASELINES — particion {split.upper()}")
    print("=" * 96)
    print(format_table(baseline_tables[split]))
    print()

BASELINES — particion TRAIN
            Model                                  Description      RMSE       MAE   MAPE      R2   n
          sarimax SARIMAX + calendar exog, walk-forward 1-step   411,494   241,982  6.56%  0.9088 889
   seasonal_naive    yhat_t = y_{t-7} (same weekday last week)   886,040   439,381 11.61%  0.5772 889
 moving_average_7              yhat_t = mean(y_{t-7}..y_{t-1}) 1,250,020 1,089,063 29.84%  0.1586 889
moving_average_28             yhat_t = mean(y_{t-28}..y_{t-1}) 1,296,740 1,133,500 31.00%  0.0945 889
      persistence                             yhat_t = y_{t-1} 1,385,175   921,712 24.03% -0.0332 889

BASELINES — particion VAL
            Model                                  Description      RMSE       MAE   MAPE     R2   n
          sarimax SARIMAX + calendar exog, walk-forward 1-step   404,153   273,467  7.31% 0.9248 196
   seasonal_naive    yhat_t = y_{t-7} (same weekday last week)   883,627   460,615 12.63% 0.6408 196
 moving_average_7             

In [12]:
# ---- Etapa 1: CARGADA (ver la justificacion sobre esta celda) ----------------------------
from src.models.lstm.final_model import (
    FINAL_MODEL_FILE,
    FINAL_SCALER_FILE,
    VAL_TEST_PREDICTIONS_FILE,
)
from src.models.lstm.oof import OOF_PREDICTIONS_FILE
from src.models.lstm.scaling import TargetScaler
from src.utils.paths import PROCESSED_DIR

import keras

stage1 = keras.models.load_model(FINAL_MODEL_FILE)
stage1.summary()

escalador = TargetScaler.load(FINAL_SCALER_FILE)
print(f"\nescalador de la etapa 1 : {type(escalador).__name__} (ajustado solo con train)")

oof = pd.read_parquet(OOF_PREDICTIONS_FILE)
fold_log = pd.read_parquet(PROCESSED_DIR / "lstm_oof_fold_log.parquet")
lstm_val_test = pd.read_parquet(VAL_TEST_PREDICTIONS_FILE)

cobertura = int(oof["has_oof"].sum())
print(f"predicciones OOF        : {cobertura} / {len(oof)} filas de entrenamiento "
      f"({cobertura / len(oof):.1%})")
print(f"predicciones val/test   : {len(lstm_val_test)} filas")

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,157 (51.40 KB)

 Trainable params: 4,385 (17.13 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 8,772 (34.27 KB)


escalador de la etapa 1 : TargetScaler (ajustado solo con train)
predicciones OOF        : 689 / 889 filas de entrenamiento (77.5%)
predicciones val/test   : 393 filas


In [13]:
# El fold 1 quedo degenerado y se excluye de la etapa 2 tras una comprobacion de cobertura;
# el registro por fold es lo que documenta esa decision.
display(fold_log)

,fold,epochs_run,best_epoch,final_train_loss,final_val_loss,mae,rmse
0,1,15,5,0.119460,0.078817,1.461905e+06,1.682893e+06
1,2,153,143,0.027096,0.050879,6.594255e+05,8.825239e+05
2,3,228,218,0.020705,0.028092,2.821220e+05,3.975123e+05
3,4,131,121,0.020752,0.009826,5.480809e+05,8.393179e+05
4,5,81,71,0.025142,0.033067,5.061476e+05,7.181744e+05


In [14]:
from src.evaluation.memoria_figures import oof_fold1_degenerate_figure, oof_folds_figure

display(oof_folds_figure(oof))
display(oof_fold1_degenerate_figure(oof, fold_log))

In [15]:
# ---- Etapa 2: RECOMPUTADA en vivo --------------------------------------------------------
from src.models.hybrid_residual.assemble_residual_dataset import assemble
from src.models.hybrid_residual.xgboost_residual import RESIDUAL_MODEL_FILE
from src.models.hybrid_residual import xgboost_residual

# residual = y_true - y_pred_oof, sobre folds 2-5 (el fold 1, degenerado, queda fuera).
residual_set = assemble()
print(f"conjunto residual : {len(residual_set)} filas  "
      f"({residual_set['date'].min():%Y-%m-%d} -> {residual_set['date'].max():%Y-%m-%d})")
print(f"folds incluidos   : {sorted(residual_set['fold'].unique().tolist())}\n")

t = time.perf_counter()
modelo_residual, tuning_residual, cols_residual = xgboost_residual.train(residual_set)
print(f"\n[{time.perf_counter() - t:.1f} s] etapa 2 reentrenada")

# El artefacto publicado esta en disco: si la rejilla eligiese hoy otra configuracion, la
# comparacion de la etapa 5 dejaria de ser una reproduccion. Se comprueba, no se supone.
params_publicados = joblib.load(RESIDUAL_MODEL_FILE)["params"]
assert tuning_residual.params == params_publicados, (
    f"hiperparametros divergentes: {tuning_residual.params} vs {params_publicados}"
)
print(f"hiperparametros identicos a models/xgboost_residual.joblib: {tuning_residual.params}")

conjunto residual : 552 filas  (2024-01-01 -> 2025-07-05)
folds incluidos   : [2, 3, 4, 5]

STAGE 2 — XGBoost on LSTM OOF residuals
  training rows        : 552
  features             : 161
  date coverage        : 2024-01-01 -> 2025-07-05


  candidates evaluated : 54
  internal holdout     : 2025-04-14 -> 2025-07-05 (83 rows; 469 used to fit)
                         ^ carved from TRAINING; not the official val split
  selected             : max_depth=5, n_estimators=300, learning_rate=0.01, min_child_weight=3
  holdout MAE / RMSE   : 380,051 / 563,793



[13.6 s] etapa 2 reentrenada
hiperparametros identicos a models/xgboost_residual.joblib: {'max_depth': 5, 'n_estimators': 300, 'learning_rate': 0.01, 'min_child_weight': 3}


In [16]:
# ---- Control XGBoost-alone: RECOMPUTADO en vivo ------------------------------------------
from src.models.hybrid_residual import xgboost_alone
from src.models.hybrid_residual.xgboost_alone import ALONE_MODEL_FILE

# Entrena sobre las 889 filas completas de train, NO sobre las 552 del subconjunto residual:
# no depende de los folds OOF, y recortarlo "por justicia" seria justo lo contrario.
t = time.perf_counter()
modelo_alone, tuning_alone, cols_alone = xgboost_alone.train(xgboost_alone.load_training_split())
print(f"\n[{time.perf_counter() - t:.1f} s] control reentrenado")

params_alone_publicados = joblib.load(ALONE_MODEL_FILE)["params"]
assert tuning_alone.params == params_alone_publicados, (
    f"hiperparametros divergentes: {tuning_alone.params} vs {params_alone_publicados}"
)
print(f"hiperparametros identicos a models/xgboost_alone.joblib: {tuning_alone.params}")

CONTROL — XGBoost trained directly on `total` (not on residuals)
  training rows        : 889  (full Phase 3 train split)
  features             : 161
  date coverage        : 2023-01-29 -> 2025-07-05


  candidates evaluated : 54
  internal holdout     : 2025-02-23 -> 2025-07-05 (133 rows; 756 used to fit)
                         ^ carved from TRAINING; not the official val split
  selected             : max_depth=5, n_estimators=100, learning_rate=0.05, min_child_weight=10
  holdout MAE / RMSE   : 275,152 / 382,713

[14.9 s] control reentrenado
hiperparametros identicos a models/xgboost_alone.joblib: {'max_depth': 5, 'n_estimators': 100, 'learning_rate': 0.05, 'min_child_weight': 10}


In [17]:
# ---- Combinacion hibrida: y_final = y_lstm + e_xgboost -----------------------------------
from src.models.hybrid_residual.combine import combine

# combine() lee el modelo residual de una ruta. Se le pasa el que acaba de entrenarse, volcado
# a un fichero temporal, para que la prediccion hibrida proceda del modelo recomputado y no del
# artefacto en models/. El fichero temporal se descarta al salir; models/ no se toca.
with tempfile.TemporaryDirectory() as tmp:
    ruta_tmp = Path(tmp) / "xgboost_residual_recomputado.joblib"
    joblib.dump(
        {"model": modelo_residual, "features": cols_residual, "params": tuning_residual.params},
        ruta_tmp,
    )
    hibrido = combine(residual_model_path=ruta_tmp)

print(f"filas hibridas      : {len(hibrido)}  "
      f"({hibrido['date'].min():%Y-%m-%d} -> {hibrido['date'].max():%Y-%m-%d})")
print(f"correccion media |e|: {hibrido['e_pred_xgb'].abs().mean():,.0f} viajeros/dia")
display(hibrido.head(3))

filas hibridas      : 393  (2025-07-06 -> 2026-08-02)
correccion media |e|: 268,210 viajeros/dia


,date,y_true,y_pred_lstm,e_pred_xgb,y_pred_hybrid,split
0,2025-07-06,2616319.0,2.482804e+06,27089.888672,2.509894e+06,val
1,2025-07-07,4985140.0,5.053655e+06,72920.718750,5.126576e+06,val
2,2025-07-08,5119565.0,5.449083e+06,-105430.625000,5.343652e+06,val


---

# 5. Comparación de resultados

La tabla maestra puntúa todos los modelos con el mismo `metrics.py` y sobre las mismas
fronteras de partición, de modo que las cifras son directamente comparables.

**Léase la columna `n`.** No todos los modelos cubren las mismas filas en todas las
particiones: la LSTM y el híbrido necesitan una ventana de 28 días y no pueden predecir las
primeras 28 filas, y los modelos derivados de OOF sólo están definidos sobre el periodo de
entrenamiento. Dentro de validación y de test, en cambio, todos cubren exactamente las mismas
filas, que es sobre lo que descansa la comparación de cabecera.

**Resultado empírico del trabajo:** el híbrido **no** supera a SARIMAX ni a `xgboost_alone` ni
en validación ni en test. La corrección residual en dos etapas no aporta sobre apuntar
directamente un modelo de árboles al problema con las mismas variables. Es el hallazgo que la
memoria discute, y esta tabla es la evidencia.

In [18]:
from src.evaluation.full_comparison import (
    assemble_predictions,
    build_comparison_frame,
    build_tables,
)

# Se reutiliza la MISMA funcion de ensamblado que produjo la tabla publicada, alimentada con
# las piezas recomputadas en memoria. Reimplementar aqui la union haria que la comparacion
# midiese la divergencia entre dos ensamblados en lugar de medir el pipeline.
anchas = assemble_predictions(
    baselines=baseline_preds,
    hybrid=hibrido,
    alone_payload={"model": modelo_alone, "features": cols_alone},
    features=features,
)
tabla_recomputada = build_comparison_frame(build_tables(anchas))

for split in ("train", "val", "test"):
    subset = tabla_recomputada[tabla_recomputada["split"] == split].drop(columns="split")
    print("=" * 96)
    titulo = f"COMPARACION MAESTRA — particion {split.upper()}"
    if split == "train":
        titulo += "   (en muestra para LSTM/XGBoost; solo referencia)"
    print(titulo)
    print("=" * 96)
    print(format_table(subset))
    print()

COMPARACION MAESTRA — particion TRAIN   (en muestra para LSTM/XGBoost; solo referencia)
            Model                              Description      RMSE       MAE   MAPE      R2   n
    xgboost_alone    XGBoost on total, full feature matrix   114,865    72,903  1.77%  0.9929 889
          sarimax SARIMAX(2,1,2)x(0,1,2,7) + calendar exog   411,494   241,982  6.56%  0.9088 889
   seasonal_naive                         yhat_t = y_{t-7}   886,040   439,381 11.61%  0.5772 889
 moving_average_7                      trailing 7-day mean 1,250,020 1,089,063 29.84%  0.1586 889
moving_average_28                     trailing 28-day mean 1,296,740 1,133,500 31.00%  0.0945 889
      persistence                         yhat_t = y_{t-1} 1,385,175   921,712 24.03% -0.0332 889

COMPARACION MAESTRA — particion VAL
            Model                              Description      RMSE       MAE   MAPE     R2   n
    xgboost_alone    XGBoost on total, full feature matrix   355,027   244,843  5.68% 0.9420

## Verificación: el cuaderno se comprueba contra los valores publicados

Cada métrica recomputada arriba se contrasta, celda a celda, con el valor publicado en
`data/processed/full_comparison.parquet` — el mismo fichero del que la memoria toma sus
cifras. Se comparan `RMSE`, `MAE`, `MAPE` y `R²` de los 8 modelos en las 3 particiones, más el
recuento `n` por igualdad exacta, y también el propio conjunto de filas: un modelo de más o de
menos es un defecto estructural que ninguna tolerancia puede expresar.

**Tolerancia: `rtol = 1e-6`** (desviación relativa). Sobre un MAE de test de ~156 000 son
0,16 viajeros/día, seis órdenes de magnitud por debajo de la precisión con la que la memoria
reporta nada. No absorbe deriva: en la máquina de referencia la desviación observada es
exactamente 0,0 en todas las celdas.

> ### Si esta celda levanta una excepción, léala antes de sacar conclusiones
>
> La tabla completa se muestra **siempre**, antes de cualquier excepción, para que se vea qué
> celda se ha desviado y cuánto.
>
> XGBoost usa `tree_method="hist"` con `n_jobs=-1`, y su histograma multihilo **suma en coma
> flotante en un orden distinto según el número de núcleos de la máquina**. En un ordenador
> distinto del de referencia es posible superar `1e-6` por esa razón y sólo por esa razón. El
> mensaje de la excepción clasifica la magnitud observada e indica explícitamente si
> corresponde a una diferencia de entorno de ejecución (desviaciones por debajo de `1e-3`
> relativo, invisibles a la precisión publicada) o a un cambio real del pipeline (que movería
> las métricas en **miles** de viajeros). Una desviación del primer tipo **no** significa que
> el trabajo no reproduzca.

In [19]:
from src.evaluation.full_comparison import (
    DEFAULT_RTOL,
    assert_within_tolerance,
    load_published,
    verify_against_published,
)

# La tabla se construye y se MUESTRA primero; el veredicto se emite despues. Si el veredicto
# levanta, la evidencia ya esta impresa encima de la excepcion.
contraste = verify_against_published(tabla_recomputada, load_published())

resumen = (
    contraste.groupby(["split", "Model"])["desviacion_rel"]
    .max()
    .reset_index()
    .rename(columns={"desviacion_rel": "desviacion_rel_maxima"})
    .sort_values(["split", "desviacion_rel_maxima"], ascending=[True, False])
)
print(f"CONTRASTE CONTRA data/processed/full_comparison.parquet  "
      f"({len(contraste)} valores, rtol={DEFAULT_RTOL:g})\n")
print("Desviacion relativa maxima por modelo y particion:")
display(resumen.reset_index(drop=True))

print("\nDetalle completo, celda a celda:")
display(contraste)

print()
print(assert_within_tolerance(contraste, rtol=DEFAULT_RTOL))

CONTRASTE CONTRA data/processed/full_comparison.parquet  (110 valores, rtol=1e-06)

Desviacion relativa maxima por modelo y particion:


,split,Model,desviacion_rel_maxima
0,test,hybrid,0.0
1,test,lstm_alone,0.0
2,test,moving_average_28,0.0
3,test,moving_average_7,0.0
4,test,persistence,0.0
5,test,sarimax,0.0
6,test,seasonal_naive,0.0
7,test,xgboost_alone,0.0
8,train,moving_average_28,0.0
9,train,moving_average_7,0.0



Detalle completo, celda a celda:


,Model,split,metrica,publicado,recomputado,desviacion_abs,desviacion_rel
0,hybrid,test,MAE,234223.295428,234223.295428,0.0,0.0
1,hybrid,test,MAPE,5.494681,5.494681,0.0,0.0
2,hybrid,test,R2,0.939208,0.939208,0.0,0.0
3,hybrid,test,RMSE,328130.542939,328130.542939,0.0,0.0
4,hybrid,test,n,197.000000,197.000000,0.0,0.0
...,...,...,...,...,...,...,...
105,xgboost_alone,val,MAE,244842.725765,244842.725765,0.0,0.0
106,xgboost_alone,val,MAPE,5.677233,5.677233,0.0,0.0
107,xgboost_alone,val,R2,0.942008,0.942008,0.0,0.0
108,xgboost_alone,val,RMSE,355026.725654,355026.725654,0.0,0.0



OK - 110/110 valores coinciden con full_comparison.parquet (rtol=1e-06; desviacion relativa maxima observada: 0.000e+00)


---

# 6. Gráficos finales

Todos los constructores viven en `src/evaluation/dashboard_figures.py` y
`src/evaluation/memoria_figures.py`. **Aquí no se define ninguna figura**: se llaman y se
muestran, de modo que las figuras de este cuaderno y las de la memoria no pueden divergir —
salen del mismo código y comparten el tema de `src/evaluation/theme.py`.

**Los PNG de la memoria no se exportan desde este kernel.** kaleido 1.x bloquea el proceso al
invocarse repetidamente dentro de Jupyter. Las 34 figuras estáticas se generan aparte, como
proceso secuencial independiente:

```bash
python -m src.evaluation.export_figures
```

In [20]:
from src.evaluation.dashboard_data import load_all_predictions

# Se carga UNA sola vez y se pasa por `predictions=` a todo lo demas: cada una de estas
# funciones releeria si no los cinco artefactos de predicciones.
predictions = load_all_predictions()
print(f"predicciones cargadas: {len(predictions)} filas, "
      f"{predictions['model'].nunique()} modelos, particiones {sorted(predictions['split'].unique())}")

predicciones cargadas: 11435 filas, 11 modelos, particiones ['test', 'train', 'val']


In [21]:
from src.evaluation.memoria_figures import (
    annual_seasonality_figure,
    error_dispersion_figure,
    operator_demand_figure,
    total_series_figure,
    weekly_seasonality_figure,
)

# --- La serie y su estructura estacional ---
display(total_series_figure(features))
display(operator_demand_figure(features))
display(weekly_seasonality_figure(features))
display(annual_seasonality_figure(features))

In [22]:
from src.evaluation.dashboard_figures import (
    comparison_figure,
    day_type_figure,
    demand_figure,
    importance_group_figure,
    importance_top_figure,
    residual_figure,
)

# --- Demanda real frente a predicha, y comparacion entre modelos ---
display(demand_figure("xgboost_alone", "test", predictions))
display(comparison_figure("test", predictions))
display(error_dispersion_figure(split="test"))

In [23]:
# --- Error de prediccion y desglose por tipo de dia ---
display(residual_figure("hybrid", "test", predictions))
display(day_type_figure("test", predictions=predictions))

In [24]:
# --- Importancia de variables del XGBoost ---
display(importance_group_figure("xgboost_residual"))
display(importance_top_figure("xgboost_alone"))

In [25]:
from src.evaluation.report_tables import comparison_table, hyperparameters_table

print("Tabla de comparacion (particion de test):")
display(comparison_table("test"))

print("Hiperparametros seleccionados:")
display(hyperparameters_table())

Tabla de comparacion (particion de test):


,Modelo,Familia,n,MAE,RMSE,MAPE (%),R²
0,ensemble_inverse_mae,Ensemble,197,139681,208339,3.11,0.9755
1,ensemble_equal,Ensemble,197,139725,208464,3.13,0.9755
2,xgboost_alone,Una etapa,197,156121,234154,3.30,0.9690
3,sarimax,Una etapa,197,163627,241147,3.85,0.9672
4,hybrid_weighted,Familia LSTM,197,220286,319447,5.17,0.9424
5,hybrid,Familia LSTM,197,234223,328131,5.49,0.9392
6,lstm_alone,Familia LSTM,197,280952,431714,6.62,0.8948
7,seasonal_naive,Baseline ingenuo,197,309817,690480,7.16,0.7308
8,persistence,Baseline ingenuo,197,900570,1390444,21.79,-0.0916
9,moving_average_7,Baseline ingenuo,197,1093679,1260492,28.13,0.1029


Hiperparametros seleccionados:


,Modelo,Filas entrenamiento,max_depth,n_estimators,learning_rate,min_child_weight
0,xgboost_residual,552,5,300,0.01,3
1,xgboost_alone,889,5,100,0.05,10


---

# Resumen

El flujo recorrido en este cuaderno, de principio a fin:

1. **Datos.** Tres fuentes crudas unificadas en 1310 días continuos (2023-01-01 → 2026-08-02),
   sin huecos, sin duplicados, sin nulos, y con la identidad
   `metro+emt+carretera+cercanias == total` verificada exactamente.
2. **Variables.** 161 columnas `feat_` en cinco bloques, con las guardias anti-fuga
   disparándose de verdad ante cada violación inyectada.
3. **Partición.** 70/15/15 cronológico (917/196/197 días), warm-up de 28 filas confinado al
   bloque de entrenamiento, 889 filas de train efectivas.
4. **Modelos.** Baselines, SARIMAX, XGBoost residual y XGBoost-alone recomputados en vivo;
   etapa 1 LSTM cargada de artefacto para preservar la identidad del modelo que produjo los
   resultados publicados.
5. **Comparación.** Tabla maestra sobre las mismas particiones y el mismo `metrics.py`,
   **contrastada contra los valores publicados** con `rtol = 1e-6` y fallo explícito.
6. **Gráficos.** Construidos por `src/evaluation/`, compartiendo tema con las figuras de la
   memoria.

**El hallazgo.** El híbrido residual en dos etapas no supera a SARIMAX ni a `xgboost_alone`.
La descomposición secuencial —LSTM para la señal, XGBoost para el residuo— no aporta sobre
apuntar directamente el modelo de árboles al problema con la misma matriz de variables. El
mejor resultado del trabajo lo da un ensamble equiponderado SARIMAX + XGBoost-alone. Ese es el
resultado que la memoria reporta y discute, y este cuaderno acaba de reproducirlo.

In [26]:
print(f"tiempo total de ejecucion del cuaderno: {time.perf_counter() - T0:.1f} s")

tiempo total de ejecucion del cuaderno: 31.4 s
